In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import random
import time
from tqdm import tqdm
import shutil

In [ ]:
def crop_eye_images_from_model_weights(model_path, subject_ids, root_dir, output_dir, device='cpu'):
    model = LightweightBBoxCNN().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])

    failed_subjects = {}
    total_time = 0.0
    total_preds = 0

    for subject_id in tqdm(subject_ids, desc="Processing Subjects"):
        subject_str = f"S_{subject_id}"
        subject_output_dir = os.path.join(output_dir, subject_str)
        os.makedirs(subject_output_dir, exist_ok=True)

        img_dir = os.path.join(root_dir, 'openEDS', 'openEDS', subject_str)
        image_paths = sorted([os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.endswith('.png')], key=lambda x: int(os.path.basename(x).split('.')[0]))

        failed_indices = []

        # Always copy image 0
        shutil.copy(image_paths[0], os.path.join(subject_output_dir, "0.png"))

        for idx in range(len(image_paths) - 1):
            img0 = Image.open(image_paths[idx]).convert('L')
            img1 = Image.open(image_paths[idx + 1]).convert('L')

            img0_t = transform(img0)
            img1_t = transform(img1)
            diff = img1_t - img0_t
            input_tensor = torch.cat((img1_t, diff), dim=0).unsqueeze(0).to(device)

            start_time = time.time()
            with torch.no_grad():
                pred_box = model(input_tensor).squeeze().cpu().numpy()
            end_time = time.time()

            total_time += end_time - start_time
            total_preds += 1

            xmin, xmax, ymin, ymax = map(int, pred_box)
            original_img = np.array(img1)

            if xmin < 0 or ymin < 0 or xmax > original_img.shape[1] or ymax > original_img.shape[0]:
                failed_indices.append(idx + 1)
                shutil.copy(image_paths[idx + 1], os.path.join(subject_output_dir, f"{idx+1}.png"))
                continue

            cropped_img = original_img[ymin:ymax, xmin:xmax]
            crop_pil = Image.fromarray(cropped_img)
            crop_pil.save(os.path.join(subject_output_dir, f"{idx+1}.png"))

        if failed_indices:
            failed_subjects[subject_str] = failed_indices

    if failed_subjects:
        with open('failed_subjects.txt', 'w') as f:
            for subject, indices in sorted(failed_subjects.items()):
                f.write(f"{subject}: {','.join(map(str, indices))}\n")
        print(f"Skipped frames due to invalid bounding boxes saved to 'failed_subjects.txt'")

    if total_preds > 0:
        avg_time = total_time / total_preds
        print(f"Average prediction time: {avg_time:.6f} seconds")

In [ ]:
model_path = r"C:\Users\omarh\Documents\GaTech\VisualTracking\scripts\preprocessing\checkpoints\cp_e50"
output_dir = r"C:\Users\omarh\OneDrive - Georgia Institute of Technology\openEDS2019\cropped"

subject_ids = get_all_subject_ids(root_dir)
crop_eye_images_from_model_weights(
    model_path=model_path,
    subject_ids=subject_ids,
    root_dir=root_dir,
    output_dir=output_dir,
    device='cuda'  # or 'cpu'
)